# Setup and Load Data

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd

# Adding to avoid casting errors when converting weather string columns to numeric
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
# df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")

# # Checkpoint as parquet (run once, then comment out)
# TEAM_DIR = "dbfs:/student-groups/Group_3_2"
# dbutils.fs.mkdirs(TEAM_DIR)
# df_otpw.write.mode("overwrite").parquet(f"{TEAM_DIR}/otpw_3m.parquet")
# print("Parquet saved.")

In [0]:
# Always read from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
df = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")

In [0]:
display(df)

# General Info and Feature Types

In [0]:
df.printSchema()

In [0]:
# Dataset dimensions
num_rows = df.count()
num_cols = len(df.columns)
print(f"Rows: {num_rows:,}")
print(f"Columns: {num_cols}")

# Handle Duplicates, Missing Data, and Leakage

## Duplicates

In [0]:
total = df.count()
distinct = df.dropDuplicates().count()
print(f"Total rows: {total}")
print(f"Distinct rows: {distinct}")
print(f"Duplicate rows: {total - distinct}")

## Missing Data for Response Variable

In [0]:
null_exprs = [
    count(
        when(col(c).isNull() | (col(c).cast("string") == ""), c)
    ).alias(c)
    for c in df.columns
]
null_counts = df.select(null_exprs).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_percentage"] = (null_counts["null_count"] / num_rows * 100).round(2)

null_counts = (
    null_counts
    .reset_index()
    .rename(columns={"index": "column_name"})
    .sort_values("null_percentage", ascending=False)
)

high_missing_cols = null_counts[null_counts["null_percentage"] > 50]
print(f"Columns with >50% missing: {len(high_missing_cols)}")
display(null_counts)

In [0]:
delayed = df.filter(col("DEP_DEL15") == 1)
on_time = df.filter(col("DEP_DEL15") == 0)
missing = df.filter(col("DEP_DEL15").isNull())

print(f"Total delayed flights: {delayed.count()}")
print(f"Total on-time flights: {on_time.count()}")
print(f"Missing DEP_DEL15: {missing.count()}")

We can see that DEP_DEL15, the variable we are interested in, has 3.02 percent missing data. This could be due to the fact that there are cancelled flights that may have null DEP_DEL15 values.

In [0]:
# Is it true that all cancelled flights will not have DEP_DELAY and DEP_DEL15?
cancelled = df.filter(col("CANCELLED") == 1)
total_cancelled = cancelled.count()
cancelled_with_delay = cancelled.filter(col('DEP_DELAY').isNotNull()).count()
cancelled_with_del15 = cancelled.filter(col('DEP_DEL15').isNotNull()).count()

print(f"Total cancelled flights: {total_cancelled} ({total_cancelled / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DELAY: {cancelled_with_delay} ({cancelled_with_delay / num_rows * 100:.2f}% of dataset)")
print(f"Cancelled with non-null DEP_DEL15: {cancelled_with_del15} ({cancelled_with_del15 / num_rows * 100:.2f}% of dataset)")

We decide to remove cancelled flights as it may introduce confounding variables on an already unbalanced dataset.

In [0]:
# Filter out cancelled flights
df = df.filter(col("CANCELLED") == 0)
num_rows = df.count()
print(f"After removing cancelled flights: {num_rows:,} rows remaining")

In [0]:
missing = df.filter(col("DEP_DEL15").isNull())
print(f"Missing DEP_DEL15: {missing.count()}")

There are no more missing data for DEP_DEL15.

In [0]:
# Class balance of DEP_DEL15
target_dist = df.groupBy("DEP_DEL15").count().toPandas()
target_dist["percentage"] = (target_dist["count"] / target_dist["count"].sum() * 100).round(2)
print(target_dist)

plt.figure(figsize=(6, 4))
plt.bar(target_dist["DEP_DEL15"].astype(str), target_dist["count"])
plt.title("DEP_DEL15 Class Distribution (3-Month OTPW)")
plt.xlabel("DEP_DEL15 (0=On-time, 1=Delayed ≥15min)")
plt.ylabel("Count")
for i, row in target_dist.iterrows():
    plt.text(i, row["count"], f"{row['percentage']}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## Other missing data explaination

A significant amount of missing data comes from the weather categories. However, we would like to use this data since we believe weather is an important predictor to weather or not a flight is delayed. Specifically, we are concerned with hourly weather categories.

We first check report type for missing data of weather. One of the reasons why there are so many missing values is that the weather dataset has multiple report types (FM-15, FM-16, FM-12, etc) that do not contain all information. For example:

* FM-15: Standard hourly temperature, wind, visibility, pressure, and precipitation observations. This is the most complete and consistent report type, making it ideal for linking to individual flights.
* FM-16: Special weather report issued when conditions change significantly (e.g., sudden storms or extreme winds). It is irregular and event-driven, so coverage is sparse.
* FM-12: Synoptic report issued every 6–12 hours with fewer variables; many fields are missing and it does not align well with hourly flight data.
* SOD (Summary of Day): Daily aggregates like total precipitation or max/min temperature; not useful for per-flight modeling.
* SHEF and SY-MT: Rare or specialized reports (hydrology or system metadata) with minimal rows and missing data.

Because of this, we focus primarily on FM-15 for modeling. In the future we can optionally including FM-16 if we want to capture extreme weather events. As for the other report tyes, we will and drop them to ensure consistent and reliable features for each flight.

In [0]:
df.groupBy("REPORT_TYPE") \
  .agg(count("*").alias("count")) \
  .display()

In [0]:
df = df.filter(col("CANCELLED") == 0)
num_rows = df.count()
print(f"After removing cancelled flights: {num_rows:,} rows remaining")

In [0]:
df.groupBy("SOURCE").count().show()

In [0]:
weather_cols = [
    "HourlyAltimeterSetting",
    "HourlyDewPointTemperature",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyPresentWeatherType",
    "HourlyPressureChange",
    "HourlyPressureTendency",
    "HourlyRelativeHumidity",
    "HourlySkyConditions",
    "HourlySeaLevelPressure",
    "HourlyStationPressure",
    "HourlyVisibility",
    "HourlyWetBulbTemperature",
    "HourlyWindDirection",
    "HourlyWindGustSpeed",
    "HourlyWindSpeed"
]

exprs = [
    (count(when(col(c).isNull(), True)) / count("*")).alias(f"{c}_missing_pct")
    for c in weather_cols
]

df.groupBy("SOURCE").agg(*exprs).display()

In the table, we can see that source 7 and 6 make up the majority of the dataset. Besides source O, which has all missing data, all other sources still have constistent data for at least a couple of the categories

In [0]:
df_weather_3m = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")
display(df_weather_3m.filter(col("SOURCE") == "7").filter(col("STATION")=="72219013874"))

Much of the missing data comes from the weather dataset. This is attributed to the variations of each station. Certain stations may not measure/have information about specific measurements. Null is not missing data, it is zero. For example, a null in hourly precipitation is 0 precipitation.

In [0]:
df_weather_3m.groupBy("REPORT_TYPE") \
  .agg(count("*").alias("count")) \
  .show()

## Leakage

# Normalization and Transformations

# Feature Engineering and Creation

# Future Plans

We are concerned with false negatives. This is the case where we predict dep_delay15 is 0 (no delay) when there is actually a delay (1). Therefore we will concern ourselves with measuring recall. We will avoid accuracy since the dataset is imbalanced; there are a lot more non-delayed flights then there are delayed ones.